In [1]:
import os
import json
import asyncio
from typing import TypedDict, Any
from urllib.parse import urlparse, urljoin

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_deepseek import ChatDeepSeek



load_dotenv()

def get_deepseek_llm(
    model: str | None = None,
    temperature: float = 0.0,
    max_tokens: int = 1500,
) -> ChatDeepSeek:
    api_key = os.getenv("DEEPSEEK_API_KEY")
    model_name = model or os.getenv("DEEPSEEK_MODEL", "deepseek-chat")

    if not api_key:
        raise RuntimeError("DEEPSEEK_API_KEY not found in environment or .env file")

    return ChatDeepSeek(
        model=model_name,
        api_key=api_key,
        temperature=temperature,
        max_tokens=max_tokens,
    )

llm = get_deepseek_llm()

response = await llm.ainvoke(
    "Расскажи анекдот про студентов Высшей школы экономики"
)
print(response.content)

Вот анекдот в тему:

Встречаются два студента Вышки. Один говорит:
— Слушай, вчера на экзамене по эконометрике такой прикол был! Препод спрашивает: «Назовите три основных допущения классической линейной регрессии». Я отвечаю: «Первое — линейность, второе — гомоскедастичность, третье — отсутствие автокорреляции». А он: «Молодец, пять!»
Второй студент вздыхает:
— А мне вчера на «Институциональной экономике» попался вопрос: «Почему в России не работают формальные институты?» Я говорю: «Потому что неформальные работают лучше». Препод: «Логично, но где эмпирика?» Я: «А вы сами вчера в очереди в столовой видели, как бабушка профессора вперед пропустили?» Поставил «отлично» и сказал, что я эмпирический гений.

Третий подходит:
— А у меня на «Теории игр» задача: «Вы и ваш друг одновременно выбираете число от 1 до 10. Кто назовет больше — тот платит за обед». Я говорю: «Равновесие Нэша — выбрать 1, чтобы минимизировать потери». Препод: «А если друг иррационален?» Я: «Тогда это уже не теория игр

In [2]:
# =========================
# Инструмент получения котировок
# =========================

from __future__ import annotations

from datetime import datetime
from typing import Any, Dict, List, Optional

from quik_python import Quik
from quik_python.data_structures import CandleInterval
from bot_config_store import BotConfigStore


def _candle_dt_to_datetime(candle_dt: Any) -> datetime:
    return datetime(
        year=int(candle_dt.year),
        month=int(candle_dt.month),
        day=int(candle_dt.day),
        hour=int(candle_dt.hour),
        minute=int(candle_dt.min),
        second=int(candle_dt.sec),
    )


def _normalize_candle(candle: Any) -> Dict[str, Any]:
    dt = _candle_dt_to_datetime(candle.datetime)
    return {
        "dt": dt,
        "open": float(candle.open),
        "close": float(candle.close),
        "high": float(candle.high),
        "low": float(candle.low),
        "volume": float(candle.volume),
    }


async def _load_intraday_with_prev_close_async(
    ticker: str,
    class_code: str,
    host: str = "localhost",
    candles_count: int = 50,
) -> List[dict]:
    """
    Возвращает точки вида:
    [
        {"label": "prev_close", "value": 312.45},
        {"label": "10:00", "value": 314.10},
        {"label": "11:00", "value": 313.70},
        ...
    ]
    """

    async with Quik(host=host) as quik:
        await quik.initialize()

        if not quik.is_service_alive():
            raise RuntimeError("QUIK service bridge is not alive")

        if not await quik.service.is_connected():
            raise RuntimeError("QUIK terminal is not connected to trading server")

        candles = await quik.candles.get_last_candles(
            class_code,
            ticker,
            CandleInterval.H1,
            candles_count,
        )

    if not candles:
        raise ValueError(f"No candles returned for {class_code}.{ticker}")

    normalized = [_normalize_candle(c) for c in candles]
    normalized.sort(key=lambda x: x["dt"])

    last_day = normalized[-1]["dt"].date()
    today_candles = [c for c in normalized if c["dt"].date() == last_day]
    prev_day_candles = [c for c in normalized if c["dt"].date() < last_day]

    if not today_candles:
        raise ValueError(f"No current-day candles found for {class_code}.{ticker}")

    prev_close: Optional[float] = None
    if prev_day_candles:
        prev_close = prev_day_candles[-1]["close"]
    else:
        # fallback, если в выборке нет предыдущего дня
        # можно либо падать, либо взять open первой текущей свечи как суррогат
        prev_close = today_candles[0]["open"]

    points: List[dict] = [{"label": "prev_close", "value": float(prev_close)}]

    for candle in today_candles:
        label = candle["dt"].strftime("%H:%M")
        points.append({
            "label": label,
            "value": float(candle["close"]),
        })

    return points


async def load_intraday_with_prev_close_async(
    quik_client: Any,
    ticker: str,
    class_code: str,
    candles_count: int = 50,
) -> List[dict]:
    candles = await quik_client.candles.get_last_candles(
        class_code,
        ticker,
        CandleInterval.H1,
        candles_count,
    )

    if not candles:
        raise ValueError(f"No candles returned for {class_code}.{ticker}")

    normalized = [_normalize_candle(c) for c in candles]
    normalized.sort(key=lambda x: x["dt"])

    last_day = normalized[-1]["dt"].date()
    today_candles = [c for c in normalized if c["dt"].date() == last_day]
    prev_day_candles = [c for c in normalized if c["dt"].date() < last_day]

    if not today_candles:
        raise ValueError(f"No current-day candles found for {class_code}.{ticker}")

    if prev_day_candles:
        prev_close = prev_day_candles[-1]["close"]
    else:
        prev_close = today_candles[0]["open"]

    points: List[dict] = [{"label": "prev_close", "value": float(prev_close)}]

    for candle in today_candles:
        points.append({
            "label": candle["dt"].strftime("%H:%M"),
            "value": float(candle["close"]),
        })

    return points


async def get_intraday_with_prev_close(
    quik_client,
    ticker: str,
    class_code: str,
):
    return await load_intraday_with_prev_close_async(
        quik_client=quik_client,
        ticker=ticker,
        class_code=class_code,
        candles_count=100,
    )

In [3]:
from __future__ import annotations

from datetime import datetime
from typing import Any, Dict, TypedDict, List, Optional
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

CLASS_TO_ASSET_TYPE = {
    "TQBR": "equity",
    "TQBS": "equity",
    "TQNL": "equity",
    "TQCB": "bond",
    "TQOS": "bond",
    "TQNO": "bond",
    "TQTF": "fund",
    "TQIF": "fund",
}

BENCHMARK_BY_ASSET_TYPE = {
    "equity": "IMOEX",
    "bond": "RGBI",
    "fund": "RGBI",
}

INDEX_CLASS_CODE = "INDX"


class BenchmarkComparisonResult(BaseModel):
    different_from_benchmark: bool = Field(
        description="True (если динамика инструмента отличается от динамики соответствующего benchmark сильнее пороговых значений) или False (если отклонение не такое значительное)"
    )
    reason: str = Field(
        description="Краткое объяснение решения на русском языке."
    )


def select_benchmark(class_code: str) -> tuple[str, str]:
    if class_code not in CLASS_TO_ASSET_TYPE:
        raise ValueError(
            f"Неизвестный class_code={class_code}. Добавь его в CLASS_TO_ASSET_TYPE."
        )
    asset_type = CLASS_TO_ASSET_TYPE[class_code]
    benchmark_ticker = BENCHMARK_BY_ASSET_TYPE[asset_type]
    return asset_type, benchmark_ticker



class BenchmarkCompareState(TypedDict, total=False):
    chat_id: int
    ticker: str
    class_code: str

    asset_type: str
    benchmark_ticker: str
    benchmark_class_code: str

    instrument_points: list[dict]
    benchmark_points: list[dict]

    instrument_returns_pct: list[float]
    benchmark_returns_pct: list[float]

    instrument_returns_labeled: list[dict]
    benchmark_returns_labeled: list[dict]
    common_labels: list[str]

    comparison_result: dict
    news_explanation_result: dict
    human_explanation: str



def _candle_dt_to_datetime(candle_dt: Any) -> datetime:
    return datetime(
        year=int(candle_dt.year),
        month=int(candle_dt.month),
        day=int(candle_dt.day),
        hour=int(candle_dt.hour),
        minute=int(candle_dt.min),
        second=int(candle_dt.sec),
    )


def _normalize_candle(candle: Any) -> Dict[str, Any]:
    dt = _candle_dt_to_datetime(candle.datetime)
    return {
        "dt": dt,
        "open": float(candle.open),
        "close": float(candle.close),
        "high": float(candle.high),
        "low": float(candle.low),
        "volume": float(candle.volume),
    }


async def get_intraday_with_prev_close(
    quik_client: Any,
    ticker: str,
    class_code: str,
    candles_count: int = 50,
) -> List[dict]:
    """
    Возвращает точки вида:
    [
        {"label": "prev_close", "value": 312.45},
        {"label": "10:00", "value": 314.10},
        {"label": "11:00", "value": 313.70},
    ]
    """

    candles = await quik_client.candles.get_last_candles(
        class_code,
        ticker,
        CandleInterval.H1,
        candles_count,
    )

    if not candles:
        raise ValueError(f"No candles returned for {class_code}.{ticker}")

    normalized = [_normalize_candle(c) for c in candles]
    normalized.sort(key=lambda x: x["dt"])

    last_day = normalized[-1]["dt"].date()
    today_candles = [c for c in normalized if c["dt"].date() == last_day]
    prev_day_candles = [c for c in normalized if c["dt"].date() < last_day]

    if not today_candles:
        raise ValueError(f"No current-day candles found for {class_code}.{ticker}")

    if prev_day_candles:
        prev_close = prev_day_candles[-1]["close"]
    else:
        prev_close = today_candles[0]["open"]

    points: List[dict] = [{"label": "prev_close", "value": float(prev_close)}]

    for candle in today_candles:
        label = candle["dt"].strftime("%H:%M")
        points.append({
            "label": label,
            "value": float(candle["close"]),
        })

    return points

def compute_labeled_neighbor_returns_pct(points: list[dict]) -> list[dict]:
    """
    Из:
    [
        {"label": "prev_close", "value": 312.45},
        {"label": "10:00", "value": 314.10},
        {"label": "11:00", "value": 313.70},
    ]

    Делает:
    [
        {"label": "10:00", "return_pct": 0.528084},
        {"label": "11:00", "return_pct": -0.127348},
    ]
    """
    if len(points) < 2:
        return []

    out: list[dict] = []
    for prev_point, curr_point in zip(points[:-1], points[1:]):
        prev_v = float(prev_point["value"])
        curr_v = float(curr_point["value"])

        if prev_v == 0:
            ret = 0.0
        else:
            ret = ((curr_v - prev_v) / prev_v) * 100.0

        out.append({
            "label": curr_point["label"],
            "return_pct": round(ret, 6),
        })

    return out


async def prepare_benchmark_comparison_data(
    quik_client: Any,
    ticker: str,
    class_code: str,
) -> dict:
    asset_type, benchmark_ticker = select_benchmark(class_code)

    instrument_points = await get_intraday_with_prev_close(
        quik_client=quik_client,
        ticker=ticker,
        class_code=class_code,
    )

    benchmark_points = await get_intraday_with_prev_close(
        quik_client=quik_client,
        ticker=benchmark_ticker,
        class_code=INDEX_CLASS_CODE,
    )

    instrument_returns_labeled = compute_labeled_neighbor_returns_pct(instrument_points)
    benchmark_returns_labeled = compute_labeled_neighbor_returns_pct(benchmark_points)

    instrument_returns_map = {
        item["label"]: item["return_pct"]
        for item in instrument_returns_labeled
    }
    benchmark_returns_map = {
        item["label"]: item["return_pct"]
        for item in benchmark_returns_labeled
    }

    common_labels = sorted(
        set(instrument_returns_map.keys()) & set(benchmark_returns_map.keys())
    )

    aligned_instrument_returns = [
        {"label": label, "return_pct": instrument_returns_map[label]}
        for label in common_labels
    ]
    aligned_benchmark_returns = [
        {"label": label, "return_pct": benchmark_returns_map[label]}
        for label in common_labels
    ]

    aligned_instrument_points = [
        point for point in instrument_points
        if point["label"] == "prev_close" or point["label"] in common_labels
    ]
    aligned_benchmark_points = [
        point for point in benchmark_points
        if point["label"] == "prev_close" or point["label"] in common_labels
    ]

    print("instrument raw tail  =", instrument_points[-10:])
    print("benchmark raw tail   =", benchmark_points[-10:])

    print("instrument labels =", [x["label"] for x in instrument_points])
    print("benchmark labels  =", [x["label"] for x in benchmark_points])
    print("common_labels     =", common_labels)

    return {
        "asset_type": asset_type,
        "benchmark_ticker": benchmark_ticker,
        "benchmark_class_code": INDEX_CLASS_CODE,

        "instrument_points": aligned_instrument_points,
        "benchmark_points": aligned_benchmark_points,

        "instrument_returns_pct": [
            item["return_pct"] for item in aligned_instrument_returns
        ],
        "benchmark_returns_pct": [
            item["return_pct"] for item in aligned_benchmark_returns
        ],

        "instrument_returns_labeled": aligned_instrument_returns,
        "benchmark_returns_labeled": aligned_benchmark_returns,
        "common_labels": common_labels,
    }


async def prepare_market_comparison_node(state: BenchmarkCompareState, runtime=None) -> dict:
    if runtime is None or not hasattr(runtime, "context"):
        raise RuntimeError("runtime.context должен содержать quik_client")

    quik_client = runtime.context.get("quik_client")
    if quik_client is None:
        raise RuntimeError("В runtime.context отсутствует quik_client")

    ticker = state["ticker"]
    class_code = state["class_code"]

    prepared = await prepare_benchmark_comparison_data(
        quik_client=quik_client,
        ticker=ticker,
        class_code=class_code,
    )
    return prepared


def compare_with_benchmark_llm_node(state: BenchmarkCompareState, runtime=None) -> dict:
    if runtime is None or not hasattr(runtime, "context"):
        raise RuntimeError("runtime.context должен содержать llm")

    llm = runtime.context.get("llm")
    if llm is None:
        raise RuntimeError("В runtime.context отсутствует llm")

    structured_llm = llm.with_structured_output(BenchmarkComparisonResult)

    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                (
                    "Ты финансовый аналитический модуль. "
                    "Тебе передан тикер, его класс, выбранный benchmark, а также уже рассчитанные временные ряды и процентные изменения между соседними точками. "
                    "Твоя задача — определить, можно ли считать, что на динамике котировок инструмента сказались существенные корпоративные новости (т.е. движение вызвано очень серьезными идиосинкратчическими факторами и критически отличается от динамики соответствующего benchmark).\n\n"
                    "Сравнивай направление движения, амплитуду и синхронность изменений: значимыми являются точечные отличия более 0.1% или накопленные за несколько часов более 0.3%"
                    "Обязательно учитывай эти порговые значения и не нарушай их (нас не интересуют расхождения в пределах шумовых погрешностей). Если отличия меньше,то считай, что отклонения нет."
                    "Будь очень консервативным, и положительный ответ давай только при очень сильных расхождениях (нам не нужны шумовые сигналы)."
                    "Верни только структурированный ответ."
                ),
            ),
            (
                "human",
                (
                    "Тикер инструмента: {ticker}\n"
                    "Класс инструмента: {class_code}\n"
                    "Тип актива: {asset_type}\n"
                    "Benchmark: {benchmark_ticker}\n\n"
                    "Точки инструмента:\n{instrument_points}\n\n"
                    "Точки benchmark:\n{benchmark_points}\n\n"
                    "Доходности инструмента между соседними точками, %:\n{instrument_returns_pct}\n\n"
                    "Доходности benchmark между соседними точками, %:\n{benchmark_returns_pct}\n"
                ),
            ),
        ]
    )

    chain = prompt | structured_llm

    result: BenchmarkComparisonResult = chain.invoke(
        {
            "ticker": state["ticker"],
            "class_code": state["class_code"],
            "asset_type": state["asset_type"],
            "benchmark_ticker": state["benchmark_ticker"],
            "instrument_points": state["instrument_points"],
            "benchmark_points": state["benchmark_points"],
            "instrument_returns_pct": state["instrument_returns_pct"],
            "benchmark_returns_pct": state["benchmark_returns_pct"],
        }
    )

    return {"comparison_result": result.model_dump()}

def build_benchmark_compare_graph():
    builder = StateGraph(BenchmarkCompareState)

    builder.add_node("prepare_market_comparison", prepare_market_comparison_node)
    builder.add_node("compare_with_benchmark_llm", compare_with_benchmark_llm_node)

    builder.add_edge(START, "prepare_market_comparison")
    builder.add_edge("prepare_market_comparison", "compare_with_benchmark_llm")
    builder.add_edge("compare_with_benchmark_llm", END)

    checkpointer = InMemorySaver()
    graph = builder.compile(checkpointer=checkpointer)
    return graph


async def run_benchmark_compare_graph(
    *,
    graph,
    quik_client: Any,
    llm: Any,
    chat_id: int,
    ticker: str,
    class_code: str,
) -> dict:
    config = {
        "configurable": {
            "thread_id": f"chat:{chat_id}:ticker:{ticker}"
        }
    }

    result = await graph.ainvoke(
        {
            "chat_id": chat_id,
            "ticker": ticker,
            "class_code": class_code,
        },
        config=config,
        context={
            "quik_client": quik_client,
            "llm": llm,
        },
    )
    return result

c:\Users\GaV\AppData\Local\Programs\Python\Python312\Lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [4]:
from __future__ import annotations

import json
from typing import Any, Optional

from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

# Предполагается, что это уже импортировано у тебя в ноутбуке:
from bot_config_store import BotConfigStore
from ingest_store import IngestStore
from kommersant_parser import refresh_and_get_kommersant_news
from telegram_reader import refresh_and_get_telegram_messages


class NewsExplanationResult(BaseModel):
    cause_found: bool = Field(description="Найдена ли правдоподобная причина отклонения")
    explanation: str = Field(description="Краткое итоговое объяснение")
    reasoning_summary: str = Field(description="Почему агент считает причину найденной или не найденной")
    checked_telegram_sources: list[str] = Field(default_factory=list, description="Список телеграм-чатов и каналов, которые были проверены")
    relevant_kommersant_items: list[dict[str, Any]] = Field(default_factory=list, description="Список новостей, которые могут объяснить изменение цены")
    relevant_telegram_items: list[dict[str, Any]] = Field(default_factory=list, description="Список сообщений телеграм, которые могут объяснить изменение цены")


class SufficiencyCheckResult(BaseModel):
    cause_found: bool = Field(description="Найдена ли правдоподобная причина")
    explanation: str = Field(description="Краткое объяснение причины или отсутствия причины")
    reasoning_summary: str = Field(description="Краткое обоснование оценки")
    relevant_kommersant_docids: list[int] = Field(default_factory=list, description="Список doc_id новостей, которые могут объяснить изменение цены")
    relevant_telegram_message_keys: list[str] = Field(default_factory=list, description="Список ключей сообщений телеграм, которые могут объяснить изменение цены")
    next_source_id: Optional[str] = Field(
        default=None,
        description="Какой telegram source_id смотреть следующим, если причина пока не найдена"
    )


def _normalize_tg_sources_for_prompt(items: list[dict[str, Any]]) -> list[dict[str, Any]]:
    out: list[dict[str, Any]] = []
    for item in items:
        is_active = bool(
            item.get("is_active", item.get("is_active", 0))
        )
        if not is_active:
            continue

        out.append(
            {
                "source_id": item.get("source_id", item.get("source_id")),
                "title": item.get("title"),
                "username": item.get("username"),
                "source_input": item.get("source_input", item.get("source_input")),
            }
        )
    return out

def _build_telegram_message_link(item: dict[str, Any]) -> Optional[str]:
    username = item.get("chat_username") or item.get("username")
    message_id = item.get("message_id") or item.get("messageid")

    if username and message_id:
        username = str(username).lstrip("@")
        return f"https://t.me/{username}/{message_id}"

    return None


def _get_active_telegram_sources_for_prompt(
    *,
    chat_id: int,
    bot_config_db_path: str,
) -> list[dict[str, Any]]:
    store = BotConfigStore(bot_config_db_path)
    raw_items = store.list_telegram_sources(chat_id)
    return _normalize_tg_sources_for_prompt(raw_items)


def _get_instrument_display_name(
    *,
    chat_id: int,
    ticker: str,
    class_code: str,
    bot_config_db_path: str,
) -> Optional[str]:
    store = BotConfigStore(bot_config_db_path)
    items = store.list_quik_instruments(chat_id)

    for item in items:
        if (
            item.get("sec_code") == ticker
            and item.get("class_code") == class_code
        ):
            return item.get("display_name")

    for item in items:
        if item.get("sec_code") == ticker:
            return item.get("display_name")

    return None


def _truncate_text(s: Optional[str], max_len: int = 2500) -> str:
    if not s:
        return ""
    s = s.strip()
    if len(s) <= max_len:
        return s
    return s[:max_len] + "\n...[truncated]"


def _prepare_kommersant_items_for_prompt(news: list[dict[str, Any]], limit: int = 12) -> list[dict[str, Any]]:
    prepared: list[dict[str, Any]] = []
    for item in news[:limit]:
        prepared.append(
            {
                "docid": item.get("docid"),
                "published_at": item.get("published_at", item.get("published_at")),
                "title": item.get("title"),
                "subtitle": item.get("subtitle"),
                "url": item.get("url"),
                "light_html": _truncate_text(item.get("light_html"), max_len=3500),
            }
        )
    return prepared


def _prepare_telegram_items_for_prompt(messages: list[dict[str, Any]], limit: int = 40) -> list[dict[str, Any]]:
    prepared: list[dict[str, Any]] = []
    for item in messages[:limit]:
        prepared_item = {
            "source_id": item.get("sourceid", item.get("source_id")),
            "chat_title": item.get("chattitle", item.get("chat_title")),
            "chat_username": item.get("chatusername", item.get("chat_username")),
            "message_id": item.get("messageid", item.get("message_id")),
            "published_at": item.get("publishedat", item.get("published_at")),
            "text": _truncate_text(item.get("text"), max_len=2000),
            "has_media": item.get("hasmedia", item.get("has_media")),
            "is_reply": item.get("isreply", item.get("is_reply")),
            "links_json": item.get("linksjson"),
        }
        prepared_item["url"] = _build_telegram_message_link(prepared_item)
        prepared.append(prepared_item)

    return prepared





def _build_news_explainer_prompt() -> ChatPromptTemplate:
    return ChatPromptTemplate.from_messages(
        [
            (
                "system",
                """Ты аналитический агент, который ищет новостное объяснение значимого отклонения цены инструмента от benchmark. Твоя цель — найти правдоподобную причину отклонения.
Останавливай поиск только если:
1. причина найдена;
2. доступные Telegram-источники закончились.

Правила работы:
1. Сначала анализируй Коммерсант.
2. После каждой новой порции данных ОБЯЗАТЕЛЬНО оценивай, достаточно ли информации для объяснения движения.
3. Если информации из Коммерсанта недостаточно, обязательно выбирай ОДИН следующий Telegram-источник из переданного списка.
4. Не предлагай уже просмотренный Telegram-источник повторно.
5. Считай объяснение достаточным только если есть конкретные факты, события, заявления, корпоративные действия, регуляторные новости, сделки, дивиденды, отчетность, санкции, аварии, судебные решения, сделки M&A, рейтинговые действия или иные драйверы, которые правдоподобно объясняют движение цены.
6. Используй как тикер, так и display_name инструмента: в новостях могут писать название компании, бренда, эмитента или разговорное имя вместо биржевого кода.
7. Если явной причины нет, так и скажи. Не выдумывай.

Если информации, чтобы понять аномальное движение цены, уже достаточно:
- cause_found=true
- next_source_id=null

Если информации, чтобы объяснит отличающееся движение цены, недостаточно:
- next_source_id должен быть одним из еще не просмотренных source_id, если такие остались.
- если источники закончились, next_source_id=null.

Если ты используешь новость Коммерсанта как часть объяснения движения цены, ОБЯЗАТЕЛЬНО добавь её docid в список `relevant_kommersant_docids` (Формат: список целых чисел, например: [123456, 123457])

Если ты используешь сообщения из Telegram для вывода explanation, ОБЯЗАТЕЛЬНО добавь их ключи в relevant_telegram_message_keys.
Для каждого релевантного сообщения Telegram сформируй строковый ключ вида: "<source_id>::<message_id>", например "tg:selfinvestor::123456". Эти ключи положи в список relevant_telegram_message_keys.
Если ни одно сообщение не релевантно, верни пустой список.
""",
            ),
            (
                "human",
                """Инструмент:
ticker = {ticker}
class_code = {class_code}
display_name = {display_name}

Результат сравнения с benchmark:
{comparison_summary}

Доступные Telegram-источники:
{telegram_sources_json}

Уже просмотренные Telegram-источники:
{checked_sources_json}

Новости Коммерсанта:
{kommersant_json}

Сообщения Telegram из уже просмотренных источников:
{telegram_messages_json}

Оцени, достаточно ли данных для объяснения отклонения.
""",
            ),
        ]
    )


async def find_news_explanation_agent_node(state: dict, runtime) -> dict:
    ctx = runtime.context
    llm: ChatOpenAI = ctx["llm"]

    ingest_db_path = ctx.get("ingest_db_path", "ingest_state.db")
    bot_config_db_path = ctx.get("bot_config_db_path", "botconfig.db")
    window_minutes = ctx.get("news_window_minutes", 12 * 60)

    chat_id = state["chat_id"]
    ticker = state["ticker"]
    class_code = state["class_code"]

    comparison_result = state.get("comparison_result", {})
    comparison_summary = json.dumps(comparison_result, ensure_ascii=False, indent=2)

    display_name = _get_instrument_display_name(
        chat_id=chat_id,
        ticker=ticker,
        class_code=class_code,
        bot_config_db_path=bot_config_db_path,
    ) or ticker

    telegram_sources = _get_active_telegram_sources_for_prompt(
        chat_id=chat_id,
        bot_config_db_path=bot_config_db_path,
    )

    prompt = _build_news_explainer_prompt()
    judge = llm.with_structured_output(SufficiencyCheckResult)

    kommersant_raw = refresh_and_get_kommersant_news(
        db_path=ingest_db_path,
        window_minutes=window_minutes,
        default_sync_minutes=window_minutes,
        overlap_minutes=10,
        fetch_article_html=True,
        only_with_html=False,
    )
    kommersant_items = _prepare_kommersant_items_for_prompt(
        kommersant_raw.get("news", []),
        limit=12,
    )

    checked_sources: list[str] = []
    telegram_messages_by_source: dict[str, list[dict[str, Any]]] = {}

    max_steps = 1 + len(telegram_sources)

    for _ in range(max_steps):
        aggregated_telegram_items: list[dict[str, Any]] = []
        for src in checked_sources:
            aggregated_telegram_items.extend(telegram_messages_by_source.get(src, []))

        messages = prompt.format_messages(
            ticker=ticker,
            class_code=class_code,
            display_name=display_name,
            comparison_summary=comparison_summary,
            telegram_sources_json=json.dumps(telegram_sources, ensure_ascii=False, indent=2),
            checked_sources_json=json.dumps(checked_sources, ensure_ascii=False, indent=2),
            kommersant_json=json.dumps(kommersant_items, ensure_ascii=False, indent=2),
            telegram_messages_json=json.dumps(aggregated_telegram_items, ensure_ascii=False, indent=2),
        )

        decision: SufficiencyCheckResult = await judge.ainvoke(messages)

        if decision.cause_found:
            relevant_kommersant_items = [
                item for item in kommersant_items
                if item.get("docid") in set(decision.relevant_kommersant_docids)
            ]

            relevant_telegram_items = []
            relevant_keys = set(decision.relevant_telegram_message_keys)
            for item in aggregated_telegram_items:
                key = f'{item.get("source_id")}::{item.get("message_id")}'
                if key in relevant_keys:
                    relevant_telegram_items.append(item)

            return {
                "news_explanation_result": NewsExplanationResult(
                cause_found=True,
                explanation=decision.explanation,
                reasoning_summary=decision.reasoning_summary,
                checked_telegram_sources=checked_sources,
                relevant_kommersant_items=relevant_kommersant_items,
                relevant_telegram_items=relevant_telegram_items,
                ).model_dump()
            }

        next_source_id = decision.next_source_id
        if not next_source_id:
            break

        if next_source_id in checked_sources:
            break

        allowed_source_ids = {
            item["source_id"]
            for item in telegram_sources
            if item.get("source_id")
        }
        if next_source_id not in allowed_source_ids:
            break

        next_source_username = [item["username"] for item in telegram_sources if item.get("source_id") == next_source_id][0]

        print("next_source_id =", next_source_id)
        print("next_source_username =", next_source_username)
        print("allowed_source_ids =", allowed_source_ids)

        tg_raw = await refresh_and_get_telegram_messages(
            source=next_source_username,
            db_path=ingest_db_path,
            default_minutes=window_minutes,
            overlap_minutes=10,
            fetch_limit=1000,
            return_minutes=window_minutes,
            only_with_media=False,
            only_replies=False,
            result_limit=200,
        )

        telegram_messages_by_source[next_source_id] = _prepare_telegram_items_for_prompt(
            tg_raw,
            limit=40,
        )
        checked_sources.append(next_source_id)

    aggregated_telegram_items: list[dict[str, Any]] = []
    for src in checked_sources:
        aggregated_telegram_items.extend(telegram_messages_by_source.get(src, []))

    final_messages = prompt.format_messages(
        ticker=ticker,
        class_code=class_code,
        display_name=display_name,
        comparison_summary=comparison_summary,
        telegram_sources_json=json.dumps(telegram_sources, ensure_ascii=False, indent=2),
        checked_sources_json=json.dumps(checked_sources, ensure_ascii=False, indent=2),
        kommersant_json=json.dumps(kommersant_items, ensure_ascii=False, indent=2),
        telegram_messages_json=json.dumps(aggregated_telegram_items, ensure_ascii=False, indent=2),
    )
    final_decision: SufficiencyCheckResult = await judge.ainvoke(final_messages)



    relevant_kommersant_items = [
        item for item in kommersant_items
        if item.get("docid") in set(final_decision.relevant_kommersant_docids)
    ]

    relevant_telegram_items = []
    relevant_keys = set(final_decision.relevant_telegram_message_keys)

    print("decision.relevant_telegram_message_keys =", decision.relevant_telegram_message_keys)
    print("aggregated telegram keys =", [f'{item.get("source_id")}::{item.get("message_id")}' for item in aggregated_telegram_items[:20]])

    for item in aggregated_telegram_items:
        key = f'{item.get("source_id")}::{item.get("message_id")}'
        if key in relevant_keys:
            relevant_telegram_items.append(item)


    print("telegram_sources =", telegram_sources)
    print("checked_sources before decision =", checked_sources)
    print("decision =", decision.model_dump())

    return {
        "news_explanation_result": NewsExplanationResult(
        cause_found=False,
        explanation=final_decision.explanation,
        reasoning_summary=final_decision.reasoning_summary,
        checked_telegram_sources=checked_sources,
        relevant_kommersant_items=relevant_kommersant_items,
        relevant_telegram_items=relevant_telegram_items,
        ).model_dump()
        }

In [5]:
def route_after_benchmark_compare(state: BenchmarkCompareState):
    comparison_result = state.get("comparison_result", {})
    if comparison_result.get("different_from_benchmark"):
        return "find_news_explanation"
    return END



In [6]:
def _build_final_explanation_prompt() -> ChatPromptTemplate:
    return ChatPromptTemplate.from_messages(
        [
            (
                "system",
                """Ты пишешь короткое пояснение для частного инвестора о том,
почему инструмент сегодня ведёт себя иначе, чем рынок.

Формат:
- 2–3 абзаца по 1–3 предложения.
- Без списков и маркеров.
- Пиши по‑русски, простым, но профессиональным языком.

Структура:
1. В первом абзаце кратко опиши, насколько сильно и на каком интервале
   инструмент отклоняется от бенчмарка — опирайся только на переданное описание сравнения.
2. Во втором абзаце кратко изложи найденное объяснение этого отклонения
   на основе новостей и Telegram.
3. Если явную причину найти не удалось, так и скажи во втором абзаце,
   но упомяни факторы, которые могли влиять (если они описаны).

Не добавляй ссылки и технические детали (LLM, БД, агенты и т.п.) — ссылки добавятся отдельно.
"""
            ),
            (
                "human",
                """Инструмент: {instrument_label}

Почему движение признано значимым (резюме сравнения с бенчмарком):
{comparison_reason}

Какие новости/факторы найдены как объяснение:
{news_explanation}

Сформулируй итоговое объяснение в виде 2–3 абзацев, как описано в инструкциях."""
            ),
        ]
    )


async def build_human_explanation_llm_node(state: dict, runtime) -> dict:
    ctx = runtime.context
    llm: ChatOpenAI = ctx["llm"]

    ticker = state.get("ticker")
    class_code = state.get("class_code")
    instrument_label = f"{ticker} ({class_code})" if ticker and class_code else (ticker or "инструмент")

    comparison = state.get("comparison_result") or {}
    news_result = state.get("news_explanation_result") or {}

    comparison_reason = (comparison.get("reason") or "").strip()
    news_explanation = (news_result.get("explanation") or "").strip()

    prompt = _build_final_explanation_prompt()
    messages = prompt.format_messages(
        instrument_label=instrument_label,
        comparison_reason=comparison_reason or "Описание сравнения с бенчмарком отсутствует.",
        news_explanation=news_explanation or "Явное новостное объяснение пока не найдено.",
    )

    reply = await llm.ainvoke(messages)
    if isinstance(reply.content, str):
        base_text = reply.content.strip()
    else:
        parts = []
        for part in reply.content:
            if isinstance(part, dict) and part.get("type") == "text":
                parts.append(part.get("text", ""))
        base_text = "".join(parts).strip()

    # Теперь кодом добавляем ссылки на Telegram (5 последних)
    relevant_tg = news_result.get("relevant_telegram_items") or []
    tail = relevant_tg[-5:]

    def _build_link(item: dict) -> str:
        username = (
            item.get("chat_username")
            or item.get("chatusername")
            or item.get("username")
        )
        message_id = item.get("message_id") or item.get("messageid")
        if username and message_id:
            return f"https://t.me/{str(username).lstrip('@')}/{message_id}"
        return ""

    link_lines: list[str] = []
    for item in tail:
        link = _build_link(item)
        if not link:
            continue
        title = item.get("chat_title") or item.get("chattitle") or item.get("source_id")
        message_id = item.get("message_id") or item.get("messageid")
        link_lines.append(f"- {title} — сообщение {message_id}: {link}")

    if link_lines:
        links_block = "Источники (Telegram):\n" + "\n".join(link_lines)
        human_text = f"{base_text}\n\n{links_block}"
    else:
        human_text = base_text

    return {
        "human_explanation": human_text,
    }

async def build_human_explanation_llm_node(state: dict, runtime) -> dict:
    ctx = runtime.context
    llm: ChatOpenAI = ctx["llm"]

    ticker = state.get("ticker")
    class_code = state.get("class_code")
    instrument_label = f"{ticker} ({class_code})" if ticker and class_code else (ticker or "инструмент")

    comparison = state.get("comparison_result") or {}
    news_result = state.get("news_explanation_result") or {}

    comparison_reason = (comparison.get("reason") or "").strip()
    news_explanation = (news_result.get("explanation") or "").strip()

    prompt = _build_final_explanation_prompt()
    messages = prompt.format_messages(
        instrument_label=instrument_label,
        comparison_reason=comparison_reason or "Описание сравнения с бенчмарком отсутствует.",
        news_explanation=news_explanation or "Явное новостное объяснение пока не найдено.",
    )

    reply = await llm.ainvoke(messages)
    if isinstance(reply.content, str):
        base_text = reply.content.strip()
    else:
        parts = []
        for part in reply.content:
            if isinstance(part, dict) and part.get("type") == "text":
                parts.append(part.get("text", ""))
        base_text = "".join(parts).strip()

    relevant_tg = news_result.get("relevant_telegram_items") or []
    relevant_kommersant_items = news_result.get("relevant_kommersant_items") or []

    def _build_tg_link(item: dict) -> str:
        username = (
            item.get("chat_username")
            or item.get("chatusername")
            or item.get("username")
        )
        message_id = item.get("message_id") or item.get("messageid")
        if username and message_id:
            return f"https://t.me/{str(username).lstrip('@')}/{message_id}"
        return ""

    telegram_link_lines: list[str] = []
    for item in relevant_tg[-5:]:
        link = _build_tg_link(item)
        if not link:
            continue
        title = (
            item.get("chat_title")
            or item.get("chattitle")
            or item.get("source_id")
            or item.get("sourceid")
            or "Telegram"
        )
        message_id = item.get("message_id") or item.get("messageid")
        telegram_link_lines.append(f"- {title} — сообщение {message_id}: {link}")

    kommersant_link_lines: list[str] = []
    for item in relevant_kommersant_items[:5]:
        title = item.get("title") or "Коммерсантъ"
        url = item.get("url") or ""
        doc_id = item.get("doc_id") or item.get("docid")

        if not url and doc_id:
            url = f"https://www.kommersant.ru/doc/{doc_id}"

        if not url:
            continue

        kommersant_link_lines.append(f"- {title}: {url}")

    source_blocks: list[str] = []

    if telegram_link_lines:
        source_blocks.append("Источники (Telegram):\n" + "\n".join(telegram_link_lines))

    if kommersant_link_lines:
        source_blocks.append("Источники (Коммерсантъ):\n" + "\n".join(kommersant_link_lines))

    if source_blocks:
        sources_block = "\n\n".join(source_blocks)
        human_text = f"{base_text}\n\n{sources_block}"
    else:
        human_text = base_text

    return {
        "human_explanation": human_text,
    }


def build_benchmark_compare_graph():
    builder = StateGraph(BenchmarkCompareState)

    builder.add_node("prepare_market_comparison", prepare_market_comparison_node)
    builder.add_node("compare_with_benchmark_llm", compare_with_benchmark_llm_node)
    builder.add_node("find_news_explanation", find_news_explanation_agent_node)
    builder.add_node("build_human_explanation_llm", build_human_explanation_llm_node)

    builder.add_edge(START, "prepare_market_comparison")
    builder.add_edge("prepare_market_comparison", "compare_with_benchmark_llm")

    builder.add_conditional_edges(
        "compare_with_benchmark_llm",
        route_after_benchmark_compare,
        {
            "find_news_explanation": "find_news_explanation",
            END: END,
        },
    )

    builder.add_edge("find_news_explanation", "build_human_explanation_llm")
    builder.add_edge("build_human_explanation_llm", END)

    checkpointer = InMemorySaver()
    graph = builder.compile(checkpointer=checkpointer)
    return graph

my_graph = build_benchmark_compare_graph()

In [7]:
'''
chat_id = 355485544
ticker = "LKOH"
class_code = "TQBR"

quik = Quik(host="localhost")
await quik.initialize()
terminal_connected = await quik.service.is_connected()
print("terminal_connected =", terminal_connected)


result = await my_graph.ainvoke(
    {
        "chat_id":chat_id,
        "ticker":ticker,
        "class_code":class_code
    },
    config= {"configurable": {"thread_id": f"benchmark:{chat_id}:{class_code}:{ticker}"}},
    context={
        "quik_client": quik,
        "llm": llm,
        "ingest_db_path": "ingest_state.db",
        "bot_config_db_path": "bot_config.db",
        "news_window_minutes": 12 * 60,
    },
)


if 'human_explanation' in result.keys():
    print(result['human_explanation'])
else:
    print("Аномалий нет")
'''

'\nchat_id = 355485544\nticker = "LKOH"\nclass_code = "TQBR"\n\nquik = Quik(host="localhost")\nawait quik.initialize()\nterminal_connected = await quik.service.is_connected()\nprint("terminal_connected =", terminal_connected)\n\n\nresult = await my_graph.ainvoke(\n    {\n        "chat_id":chat_id,\n        "ticker":ticker,\n        "class_code":class_code\n    },\n    config= {"configurable": {"thread_id": f"benchmark:{chat_id}:{class_code}:{ticker}"}},\n    context={\n        "quik_client": quik,\n        "llm": llm,\n        "ingest_db_path": "ingest_state.db",\n        "bot_config_db_path": "bot_config.db",\n        "news_window_minutes": 12 * 60,\n    },\n)\n\n\nif \'human_explanation\' in result.keys():\n    print(result[\'human_explanation\'])\nelse:\n    print("Аномалий нет")\n'

In [8]:
import asyncio
from quik_python import Quik
from telegram_control import TelegramControlBot
from bot_config_store import BotConfigStore

async def run_graph_for_instrument(
    *,
    my_graph,
    quik,
    llm,
    chat_id: int,
    ticker: str,
    class_code: str,
):
    result = await my_graph.ainvoke(
        {
            "chat_id": chat_id,
            "ticker": ticker,
            "class_code": class_code,
        },
        config={"configurable": {"thread_id": f"benchmark:{chat_id}:{class_code}:{ticker}"}},
        context={
            "quik_client": quik,
            "llm": llm,
            "ingest_db_path": "ingest_state.db",
            "bot_config_db_path": "bot_config.db",
            "news_window_minutes": 12 * 60,
        },
    )
    return result

async def run_chat_scan_once(
    *,
    bot,
    my_graph,
    quik,
    llm,
    chat_id: int,
):
    settings = bot.store.get_chat_settings(chat_id)
    if not settings.monitoring_enabled:
        print(f"[chat {chat_id}] monitoring disabled")
        return

    instruments = bot.store.list_quik_instruments(chat_id)

    active_instruments = [
        x for x in instruments
        if bool(x.get("is_active", x.get("isactive", 0)))
    ]

    if not active_instruments:
        print(f"[chat {chat_id}] no active QUIK instruments")
        return

    print(f"[chat {chat_id}] scanning {len(active_instruments)} instruments")

    for item in active_instruments:
        ticker = item.get("sec_code", item.get("seccode"))
        class_code = item.get("class_code", item.get("classcode"))

        if not ticker or not class_code:
            print(f"[chat {chat_id}] skip malformed instrument: {item}")
            continue

        try:
            print(f"[chat {chat_id}] run graph for {class_code}.{ticker}")

            result = await run_graph_for_instrument(
                my_graph=my_graph,
                quik=quik,
                llm=llm,
                chat_id=chat_id,
                ticker=ticker,
                class_code=class_code,
            )

            if "human_explanation" in result:
                text = f"📉 {class_code}.{ticker}\n\n{result['human_explanation']}"
                bot.send_message(chat_id, text)
                print(f"[chat {chat_id}] alert sent for {class_code}.{ticker}")
            else:
                print(f"[chat {chat_id}] no anomaly for {class_code}.{ticker}")

        except Exception as e:
            print(f"[chat {chat_id}] error for {class_code}.{ticker}: {type(e).__name__}: {e}")


async def monitor_loop(
    *,
    bot,
    my_graph,
    quik,
    llm,
    interval_seconds: int = 300,
):
    while True:
        try:
            print("=== monitor tick ===")

            chat_ids = bot.store.list_monitored_chat_ids()

            for chat_id in chat_ids:
                settings = bot.store.get_chat_settings(chat_id)
                if not settings.monitoring_enabled:
                    print(f"[chat {chat_id}] monitoring disabled")
                    continue

                instruments = bot.store.list_quik_instruments(chat_id)
                active_instruments = [
                    x for x in instruments
                    if bool(x.get("is_active", x.get("isactive", 0)))
                ]

                if not active_instruments:
                    print(f"[chat {chat_id}] no active instruments")
                    continue

                print(f"[chat {chat_id}] scanning {len(active_instruments)} instruments")

                for item in active_instruments:
                    ticker = item.get("sec_code", item.get("seccode"))
                    class_code = item.get("class_code", item.get("classcode"))

                    if not ticker or not class_code:
                        print(f"[chat {chat_id}] skip malformed instrument: {item}")
                        continue

                    try:
                        result = await my_graph.ainvoke(
                            {
                                "chat_id": chat_id,
                                "ticker": ticker,
                                "class_code": class_code,
                            },
                            config={"configurable": {"thread_id": f"benchmark:{chat_id}:{class_code}:{ticker}"}},
                            context={
                                "quik_client": quik,
                                "llm": llm,
                                "ingest_db_path": "ingest_state.db",
                                "bot_config_db_path": "bot_config.db",
                                "news_window_minutes": 12 * 60,
                            },
                        )

                        if "human_explanation" in result:
                            text = f"📉 {class_code}.{ticker}\n\n{result['human_explanation']}"
                            bot.send_message(chat_id, text)
                            print(f"[chat {chat_id}] sent alert for {class_code}.{ticker}")
                        else:
                            print(f"[chat {chat_id}] no anomaly for {class_code}.{ticker}")

                    except Exception as e:
                        print(f"[chat {chat_id}] error for {class_code}.{ticker}: {type(e).__name__}: {e}")

        except Exception as e:
            print("monitor_loop error:", type(e).__name__, e)

        print(f"sleep {interval_seconds} sec")
        await asyncio.sleep(interval_seconds)

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(".env")

BOT_TOKEN = os.getenv("BOT_TOKEN")
if not BOT_TOKEN:
    raise RuntimeError("BOT_TOKEN is not set in .env")


quik = Quik(host="localhost")
await quik.initialize()
terminal_connected = await quik.service.is_connected()
print("QUIK terminal_connected =", terminal_connected)

bot = TelegramControlBot(
    bot_token=BOT_TOKEN,
    db_path="bot_config.db",
    quik_client=quik,
)

bot_task = asyncio.create_task(bot.start())

monitor_task = asyncio.create_task(
    monitor_loop(
        bot=bot,
        my_graph=my_graph,
        quik=quik,
        llm=llm,
        interval_seconds=300,
    )
)

await asyncio.gather(bot_task, monitor_task)

QUIK terminal_connected = True
=== monitor tick ===
[chat 355485544] scanning 8 instruments


OperationalError: database is locked

instrument raw tail  = [{'label': '09:00', 'value': 323.71}, {'label': '10:00', 'value': 324.0}, {'label': '11:00', 'value': 324.32}, {'label': '12:00', 'value': 324.18}, {'label': '13:00', 'value': 323.86}, {'label': '14:00', 'value': 322.84}, {'label': '15:00', 'value': 320.23}, {'label': '16:00', 'value': 319.5}, {'label': '17:00', 'value': 319.91}, {'label': '18:00', 'value': 320.06}]
benchmark raw tail   = [{'label': '09:00', 'value': 2546.61}, {'label': '10:00', 'value': 2540.46}, {'label': '11:00', 'value': 2540.56}, {'label': '12:00', 'value': 2534.93}, {'label': '13:00', 'value': 2534.84}, {'label': '14:00', 'value': 2525.16}, {'label': '15:00', 'value': 2498.59}, {'label': '16:00', 'value': 2483.07}, {'label': '17:00', 'value': 2484.07}, {'label': '18:00', 'value': 2493.7}]
instrument labels = ['prev_close', '06:00', '07:00', '08:00', '09:00', '10:00', '11:00', '12:00', '13:00', '14:00', '15:00', '16:00', '17:00', '18:00']
benchmark labels  = ['prev_close', '06:00', '09:00', 

[chat 355485544] no anomaly for TQBR.SBER
instrument raw tail  = [{'label': '09:00', 'value': 302.8}, {'label': '10:00', 'value': 303.2}, {'label': '11:00', 'value': 303.02}, {'label': '12:00', 'value': 301.96}, {'label': '13:00', 'value': 301.88}, {'label': '14:00', 'value': 301.48}, {'label': '15:00', 'value': 298.84}, {'label': '16:00', 'value': 297.94}, {'label': '17:00', 'value': 297.74}, {'label': '18:00', 'value': 297.94}]
benchmark raw tail   = [{'label': '09:00', 'value': 2546.61}, {'label': '10:00', 'value': 2540.46}, {'label': '11:00', 'value': 2540.56}, {'label': '12:00', 'value': 2534.93}, {'label': '13:00', 'value': 2534.84}, {'label': '14:00', 'value': 2525.16}, {'label': '15:00', 'value': 2498.59}, {'label': '16:00', 'value': 2483.07}, {'label': '17:00', 'value': 2484.07}, {'label': '18:00', 'value': 2493.64}]
instrument labels = ['prev_close', '06:00', '07:00', '08:00', '09:00', '10:00', '11:00', '12:00', '13:00', '14:00', '15:00', '16:00', '17:00', '18:00']
benchmark 

In [1]:
from e_disclosure_parser import get_e_disclosure_company_id, search_e_disclosure_companies, get_e_disclosure_events, extract_event_card_by_selenium


companies = search_e_disclosure_companies("самолет")

for c in companies:
    print(
        c["id"],
        c["name"],
        c.get("region"),
        c.get("branch"),
        c.get("lastActivity"),
        c.get("docCount"),
    )

9357 АО "ГСС" Москва Машиностроение 2020-02-14T16:41:00.003 0
31889 АО "ОАК - ТС" Москва Транспорт 2016-10-03T11:54:58.383 0
20333 АО "РСК "МиГ" Москва Машиностроение 2016-01-12T11:16:11.52 73
18796 ОАО "ССК "КИМО" Архангельская область Строительство 2011-01-12T14:32:16.623 0
2227 ПАО "ВАСО" Воронежская область Машиностроение 2021-11-01T10:05:28.037 335
36419 ПАО «ГК «Самолет» Москва Холдинги и управляющие компании 2026-06-11T22:52:26.013 317


In [2]:
company_id = 38025
year = 2026

events = get_e_disclosure_events(company_id=company_id, year=year)

for event in events:
    print(event)

{'event_name': 'Выплаченные доходы или иные выплаты, причитающиеся владельцам ценных бумаг эмитента', 'pseudo_guid': 'WLQKTypWJUSXaap8hz7ZWQ-B-B', 'pub_date': '2026-06-09T11:58:59.677', 'event_url': 'https://e-disclosure.ru/portal/event.aspx?EventId=WLQKTypWJUSXaap8hz7ZWQ-B-B'}
{'event_name': 'Выплаченные доходы или иные выплаты, причитающиеся владельцам ценных бумаг эмитента', 'pseudo_guid': '3WsDTpc-AzEyiRBt-CKtLosQ-B-B', 'pub_date': '2026-06-08T17:57:30.85', 'event_url': 'https://e-disclosure.ru/portal/event.aspx?EventId=3WsDTpc-AzEyiRBt-CKtLosQ-B-B'}
{'event_name': 'Выплаченные доходы или иные выплаты, причитающиеся владельцам ценных бумаг эмитента', 'pseudo_guid': 'E1YPI3AhKkWXNK2Mt4H7eQ-B-B', 'pub_date': '2026-06-02T14:39:54.953', 'event_url': 'https://e-disclosure.ru/portal/event.aspx?EventId=E1YPI3AhKkWXNK2Mt4H7eQ-B-B'}
{'event_name': 'Выплаченные доходы или иные выплаты, причитающиеся владельцам ценных бумаг эмитента', 'pseudo_guid': 'z80rSooB206B5ZesUslhPw-B-B', 'pub_date': '

In [5]:
data = extract_event_card_by_selenium(
    event_id="3WsDTpc-AzEyiRBt-CKtLosQ-B-B",
    headless=True,
)

print(data["company_name"])
print(data["event_title"])
print(data["published_at"])
print(data["message_text"])

ООО «Лизинг-Трейд»
Выплаченные доходы или иные выплаты, причитающиеся владельцам ценных бумаг эмитента
08.06.2026 17:57
Выплаченные доходы или иные выплаты, причитающиеся владельцам ценных бумаг эмитента

1. Общие сведения
1.1. Полное фирменное наименование (для коммерческой организации) или наименование (для некоммерческой организации) эмитента: Общество с ограниченной ответственностью «Лизинг-Трейд»
1.2. Адрес эмитента, указанный в едином государственном реестре юридических лиц: 420021, Татарстан респ., г.о. город Казань, г. Казань, ул. Галиаскара Камала, д. 41, офис 406
1.3. Основной государственный регистрационный номер (ОГРН) эмитента (при наличии): 1051622076330
1.4. Идентификационный номер налогоплательщика (ИНН) эмитента (при наличии): 1655096633
1.5. Уникальный код эмитента, присвоенный Банком России: 00506-R
1.6. Адрес страницы в сети "Интернет", используемой эмитентом для раскрытия информации: https://www.e-disclosure.ru/portal/company.aspx?id=38025; https://www.e-disclosure